# 08  Land Registry property signals (a strong company-number source)

NB06 added contract wins and NB07 added hiring demand. This notebook adds a third signal, and it
is the cleanest one yet: **which of our companies own commercial property in England and Wales.**

Owning property is a useful commercial signal. It points to an asset base, and a firm with
property is a natural fit for commercial lending, refinancing or a mortgage conversation. The data
comes from HM Land Registry's *Commercial and Corporate Ownership Data* (CCOD), which lists every
company that is the registered legal owner of property in England and Wales.

Why this source is better than the last two: **CCOD gives us the company's registration number
directly.** Contracts Finder and Adzuna mostly gave us a name, so we had to match carefully by
name and postcode. Here most rows carry a Companies House number, which is an exact, reliable join
to our spine. The matching ladder still runs for the rows that only have a name, but it does much
less of the work.

## Getting the CCOD file
The data is free but licensed. You have a **Land Registry API key**, so the notebook pulls the
latest file for you: it asks for the key when you run Section 2 (paste it in, it is not saved
anywhere), then downloads and unzips the CCOD file automatically.

No key handy? Leave the prompt blank and it uses any `CCOD_FULL_YYYY_MM.csv` you have already put in
your `MyDrive/Lloyds` folder (download one from
https://use-land-property-data.service.gov.uk/datasets/ccod after a free sign-in).

Please do not commit the CSV to GitHub; like the other data it is regenerated on re-run, so it stays
out of the repo.

## How to run
Same as the others. Run NB05 first so `lloyds.duckdb` exists, put the CCOD CSV in `MyDrive/Lloyds`,
then run this notebook top to bottom. It updates the same database.

## 1. Install and import

Same as the other notebooks: makes sure DuckDB and RapidFuzz are available, downloading them if they are not already there.

In [ ]:
import sys, subprocess
for pkg in ["duckdb", "rapidfuzz"]:
    try:
        __import__(pkg)
    except ImportError:
        subprocess.run([sys.executable, "-m", "pip", "install", "-q", pkg], check=True)

Switches the tools on. RapidFuzz is the name-comparison tool we use for the few rows that only give a company name.

In [ ]:
import duckdb
import pandas as pd
import requests
import re
from collections import defaultdict
from datetime import datetime, timezone
from rapidfuzz import process, fuzz

print("duckdb", duckdb.__version__, "| pandas", pd.__version__)

## 2. Locate the database and the CCOD file
Find the `lloyds.duckdb` that NB05 built, and find the CCOD CSV you downloaded. In Colab both live
in the same Drive folder; locally the database is under `data/processed` and the CSV can sit next
to it. The database must already exist (run NB05 first).

This box finds the database the first notebook built and looks for the CCOD file you saved. If the CCOD file is missing it stops and reminds you how to download it. `CCOD_MAX_ROWS` is a dial: leave it as `None` for the whole file, or set a number like 200000 for a quick trial run.

In [ ]:
from pathlib import Path

try:
    import google.colab  # noqa: F401
    IN_COLAB = True
except ImportError:
    IN_COLAB = False

if IN_COLAB:
    from google.colab import drive
    drive.mount("/content/drive")
    WORK_DIR = Path("/content/drive/MyDrive/Lloyds")     # same folder as NB05
    DB_PATH = WORK_DIR / "lloyds.duckdb"
else:
    WORK_DIR = Path("..").resolve() / "data" / "processed"
    DB_PATH = WORK_DIR / "lloyds.duckdb"

assert DB_PATH.exists(), f"lloyds.duckdb not found at {DB_PATH}. Run NB05 first."
print("DB:", DB_PATH)

CCOD_MAX_ROWS = None        # None = whole file; set e.g. 200000 for a quick trial
NOW = datetime.now(timezone.utc).isoformat(timespec="seconds")

# get the CCOD file. Option A: pull the latest with your Land Registry API key (paste it when
# prompted; it is not saved anywhere). Option B: leave the prompt blank and it uses a CCOD csv you
# have already put in the folder.
import io, zipfile, getpass
API_BASE = "https://use-land-property-data.service.gov.uk/api/v1"

def download_ccod_via_api(work_dir):
    key = getpass.getpass("Land Registry API key (blank = use a file already in the folder): ").strip()
    if not key:
        return None
    hdr = {"Authorization": key, "Accept": "application/json"}
    meta = requests.get(f"{API_BASE}/datasets/ccod", headers=hdr, timeout=60).json()
    resources = meta.get("result", {}).get("resources", []) if isinstance(meta.get("result"), dict) else []
    fulls = sorted(r.get("file_name", "") for r in resources if "FULL" in r.get("file_name", "").upper())
    if not fulls:
        raise RuntimeError(f"API listed no FULL file. Full response: {meta}")
    fname = fulls[-1]
    link = requests.get(f"{API_BASE}/datasets/ccod/{fname}", headers=hdr, timeout=60).json()
    url = (link.get("result") or {}).get("download_url")
    if not url:
        raise RuntimeError(f"No download_url for {fname}. Full response: {link}")
    print("downloading", fname, "...")
    blob = requests.get(url, timeout=600).content
    if fname.lower().endswith(".zip"):
        with zipfile.ZipFile(io.BytesIO(blob)) as z:
            csv_name = next(n for n in z.namelist() if n.lower().endswith(".csv"))
            z.extract(csv_name, work_dir)
            return work_dir / csv_name
    out = work_dir / fname
    out.write_bytes(blob)
    return out

CCOD_PATH = download_ccod_via_api(WORK_DIR)
if CCOD_PATH is None:
    files = sorted(WORK_DIR.glob("CCOD*.csv")) + sorted(WORK_DIR.glob("ccod*.csv"))
    if not files:
        raise FileNotFoundError(
            f"No API key given and no CCOD csv found in {WORK_DIR}.\n"
            "Either paste your API key at the prompt, or download CCOD_FULL_YYYY_MM.csv from\n"
            "  https://use-land-property-data.service.gov.uk/datasets/ccod\n"
            f"and drop it into {WORK_DIR}.")
    CCOD_PATH = files[-1]
print("CCOD file:", CCOD_PATH.name)

## 3. Helpers
The same company-number and name cleaners as NB05 and NB06, so matching uses identical normalisation. Plus a small reader that pulls a postcode out of an address line.

This box recreates the same number and name cleaners as before, so a company matches the same way everywhere. The postcode reader pulls a UK postcode out of the owner's address text, which is how we confirm a name match is the right company.

In [ ]:
def clean_company_number(value):
    if value is None:
        return None
    s = str(value).strip().upper()
    if s == "" or s == "NAN":
        return None
    if s.isdigit():
        return s.zfill(8)     # pure numbers get padded to 8 digits, matching the spine
    return s                  # letter-prefixed numbers (OC, SC, NI, FC ...) are kept as-is

_SUFFIXES = [
    "LIMITED", "LTD", "PLC", "PUBLIC LIMITED COMPANY", "LLP",
    "LIMITED LIABILITY PARTNERSHIP", "LP", "CIC", "CIO",
    "COMPANY", "CO", "AND", "THE",
]
_SUFFIX_RE = re.compile(r"\b(" + "|".join(_SUFFIXES) + r")\b")

def normalise_name(name):
    if name is None:
        return None
    s = str(name).upper()
    s = re.sub(r"[^A-Z0-9 ]", " ", s)
    s = _SUFFIX_RE.sub(" ", s)
    s = re.sub(r"\s+", " ", s).strip()
    return s or None

def normalise_postcode(pc):
    if pc is None:
        return None
    s = re.sub(r"\s+", "", str(pc).upper())
    return s or None

# an owner's address in CCOD is free text (e.g. "Sixth Floor, 6 Chesterfield Gardens, London W1J 5BQ").
# pull the postcode out by its recognisable UK shape.
_UK_PC_RE = re.compile(r"\b([A-Z]{1,2}[0-9][A-Z0-9]?\s*[0-9][A-Z]{2})\b")
def extract_postcode(text):
    if text is None or (isinstance(text, float)):
        return None
    found = _UK_PC_RE.findall(str(text).upper())
    return found[-1] if found else None

print("helpers ok")

## 4. Load the spine and build the match indexes
We pull the company number, normalised name, and postcode for every company, then build two
lookups: an exact-name index, and first-word blocks so the fuzzy step only compares names that
start with the same word. This keeps name matching fast even on the full dataset.

This box reads the master company list and builds two quick lookup tools: one finds a company by its exact simplified name, the other groups companies by their first word so the fuzzy matching later stays fast.

In [ ]:
con = duckdb.connect(str(DB_PATH))
spine = con.execute("SELECT company_number, name_norm, postcode FROM companies").df()
print(f"spine: {len(spine):,} companies")

spine_numbers = set(spine["company_number"])
by_name = defaultdict(list)     # name_norm -> [(company_number, postcode), ...]
blocks  = defaultdict(list)     # first word -> [(name_norm, company_number, postcode), ...]
for cn, nm, pc in zip(spine["company_number"], spine["name_norm"], spine["postcode"]):
    if not nm:
        continue
    pcn = normalise_postcode(pc)
    by_name[nm].append((cn, pcn))
    blocks[nm.split(" ")[0]].append((nm, cn, pcn))
print(f"exact-name keys: {len(by_name):,} | first-word blocks: {len(blocks):,}")

## 5. The matching ladder
Most CCOD rows carry a Companies House registration number, which is an exact join and needs no
guessing. For the rows that only have a name, we fall back to name matching confirmed by postcode,
exactly as in NB06, so a small firm is never confused with a big national company of a similar name.

Tiers, strongest first:
1. `company_number` (1.0) - the row carries a Companies House number that is in our dataset.
2. `name_exact_postcode` (0.95) - exact normalised name and the owner's postcode agrees.
3. `name_fuzzy_postcode` (~0.8-0.9) - RapidFuzz match above the cutoff, postcode also agrees.
4. `name_exact_unconfirmed` (0.6) - exact name, but no usable postcode to confirm with.

This box defines the matching ladder. For each property owner it tries the most reliable method first: the company registration number if there is one, then an exact name with a matching postcode, then a close fuzzy name with a matching postcode. The postcode check stops us mixing up two firms that share a name.

In [ ]:
FUZZY_CUTOFF = 90

def _postcode_match(a, b):
    return bool(a and b and a == b)

def match_proprietor(name, reg_no, owner_postcode=None):
    # 1. exact Companies House number on the row (most reliable, needs no postcode)
    cn = clean_company_number(reg_no)
    if cn and cn in spine_numbers:
        return cn, 1.0, "company_number"

    nm = normalise_name(name)
    if not nm:
        return None, 0.0, "no_match"
    own_pc = normalise_postcode(owner_postcode)

    # 2. exact normalised name, confirmed by postcode where we have one
    if nm in by_name:
        cands = by_name[nm]
        confirmed = [c for c, pc in cands if _postcode_match(own_pc, pc)]
        if confirmed:
            return confirmed[0], 0.95, "name_exact_postcode"
        if own_pc:
            return None, 0.0, "name_exact_postcode_mismatch"   # same name, wrong place
        if len(cands) == 1:
            return cands[0][0], 0.6, "name_exact_unconfirmed"
        return None, 0.0, "name_exact_ambiguous"

    # 3. fuzzy within the first-word block, only when the postcode also agrees
    if own_pc:
        bucket = blocks.get(nm.split(" ")[0])
        if bucket:
            choices = [b[0] for b in bucket]
            for _, score, idx in process.extract(
                    nm, choices, scorer=fuzz.WRatio, score_cutoff=FUZZY_CUTOFF, limit=5):
                if _postcode_match(own_pc, bucket[idx][2]):
                    return bucket[idx][1], round(score / 100 * 0.9, 3), "name_fuzzy_postcode"

    return None, 0.0, "no_match"

# sanity check: a made-up number that is not in the spine should not match
print(match_proprietor("SOME COMPANY LTD", "ZZ999999"))

## 6. Read the Land Registry CCOD file and tidy it
Each CCOD row is one property title and can list up to four owners, spread across many columns. We
reshape it into a simple long list of one row per owner, keeping the property details, the owner's
name, their registration number, and a postcode read from the owner's address.

This box reads the CCOD file and turns its wide layout (up to four owners per property, each in its own set of columns) into a tidy list of one row per owner. For each owner it keeps the property address and region, the price paid, the owner's name and registration number, and the owner's postcode read from their address.

In [ ]:
base_cols = ["Title Number", "Tenure", "Property Address", "Region",
             "Postcode", "Price Paid", "Date Proprietor Added"]
prop_cols = []
for i in (1, 2, 3, 4):
    prop_cols += [f"Proprietor Name ({i})", f"Company Registration No. ({i})",
                  f"Proprietorship Category ({i})",
                  f"Proprietor ({i}) Address (1)", f"Proprietor ({i}) Address (2)",
                  f"Proprietor ({i}) Address (3)"]
wanted = set(base_cols + prop_cols)

# read only the columns we need, as text, so large files fit in Colab memory
raw = pd.read_csv(CCOD_PATH, dtype=str, usecols=lambda c: c in wanted,
                  nrows=CCOD_MAX_ROWS, na_values=[""], keep_default_na=False)
print(f"CCOD rows (property titles): {len(raw):,}")

frames = []
for i in (1, 2, 3, 4):
    name_col = f"Proprietor Name ({i})"
    if name_col not in raw.columns:
        continue
    sub = raw[raw[name_col].notna()].copy()
    if sub.empty:
        continue
    owner_addr = (sub[f"Proprietor ({i}) Address (1)"].fillna("") + " "
                  + sub[f"Proprietor ({i}) Address (2)"].fillna("") + " "
                  + sub[f"Proprietor ({i}) Address (3)"].fillna(""))
    frames.append(pd.DataFrame({
        "title_number": sub["Title Number"],
        "tenure": sub["Tenure"],
        "property_address": sub["Property Address"],
        "region": sub["Region"],
        "price_paid": sub["Price Paid"],
        "date_added": sub["Date Proprietor Added"],
        "proprietor_name": sub[name_col],
        "company_reg_no": sub[f"Company Registration No. ({i})"],
        "proprietorship_category": sub[f"Proprietorship Category ({i})"],
        "owner_postcode": owner_addr.map(extract_postcode),
    }))

owners = pd.concat(frames, ignore_index=True)
print(f"owner rows (one per company on a property): {len(owners):,}")
print(f"  with a registration number: {owners['company_reg_no'].notna().sum():,}")
print(f"  with a readable owner postcode: {owners['owner_postcode'].notna().sum():,}")
owners.head(3)

## 7. Match owners to the spine and write owns_property signals
First the fast, exact part: match on the registration number in one vectorised step. Then run the
smaller leftover set (owners without an in-dataset registration number) through the name ladder.
Everything that matches a company in our dataset is written into `signals`. We clear any previous
Land Registry rows first so re-runs do not double count.

This box does the matching in two parts. First it matches on the company registration number in one quick step, which handles most rows. Then it runs the smaller leftover set through the name and postcode ladder. The matches are saved into the signals table as property-ownership events, clearing old Land Registry rows first so re-running does not double count.

In [ ]:
# part 1: fast exact match on the registration number
owners["clean_reg"] = owners["company_reg_no"].map(clean_company_number)
in_spine = owners["clean_reg"].isin(spine_numbers)

reg_matched = owners[in_spine].copy()
reg_matched["company_number"] = reg_matched["clean_reg"]
reg_matched["confidence"] = 1.0
reg_matched["method"] = "company_number"

# part 2: name ladder for the leftover rows only (keeps the whole file quick to process)
residual = owners[~in_spine]
name_rows = []
for r in residual.itertuples(index=False):
    cn, conf, method = match_proprietor(r.proprietor_name, r.company_reg_no, r.owner_postcode)
    if cn:
        d = r._asdict()
        d.update(company_number=cn, confidence=conf, method=method)
        name_rows.append(d)
name_matched = pd.DataFrame(name_rows) if name_rows else pd.DataFrame(columns=reg_matched.columns)

matched = pd.concat([reg_matched, name_matched], ignore_index=True)

# tidy types: value as a number, date as a real date (CCOD dates are day-month-year)
matched["value"] = pd.to_numeric(matched["price_paid"], errors="coerce")
matched["signal_date"] = pd.to_datetime(matched["date_added"], errors="coerce",
                                        dayfirst=True).dt.date
matched["detail"] = matched["property_address"].astype(str).str.slice(0, 200)
matched["signal_type"] = "owns_property"
matched["source"] = "land_registry_ccod"
matched["retrieved_at"] = NOW

print(f"owner rows matched to a company in the dataset: {len(matched):,} of {len(owners):,}")
print("\nby method:")
print(matched["method"].value_counts().to_string())

con.execute("DELETE FROM signals WHERE source = 'land_registry_ccod'")
ins = matched[["company_number", "signal_type", "signal_date", "value", "detail",
               "source", "confidence", "retrieved_at"]]
con.register("tmp_sig", ins)
con.execute("""INSERT INTO signals
               SELECT company_number, signal_type, signal_date, value, detail,
                      source, confidence, retrieved_at
               FROM tmp_sig""")
con.unregister("tmp_sig")
print(f"\nwritten to signals: {len(ins):,} rows")

## 8. Summary and a look at the result
How many companies in the dataset now own property, and a sample of the biggest owners by number of properties.

This box counts how many companies now carry a property signal and lists the ones that own the most properties.

In [ ]:
n_companies = con.execute(
    "SELECT count(DISTINCT company_number) FROM signals WHERE source='land_registry_ccod'"
).fetchone()[0]
total = con.execute("SELECT count(*) FROM companies").fetchone()[0]
print(f"companies that own property: {n_companies:,} of {total:,} ({n_companies/total:.2%})")

sample = con.execute("""
    SELECT c.company_name, c.sector,
           count(*)         AS properties,
           sum(s.value)     AS total_price_paid
    FROM signals s JOIN companies c ON c.company_number = s.company_number
    WHERE s.source = 'land_registry_ccod'
    GROUP BY c.company_name, c.sector
    ORDER BY properties DESC
    LIMIT 10
""").df()
sample

## 9. Visualise the matching and the signals
Four pictures: how the rows narrow down to a confident match, which method did the matching, which companies own the most properties, and how the matched properties spread across the country.

This box draws four charts: the funnel from owner rows down to matches, the matches by method, the top companies by number of properties, and the matched properties by region.

In [ ]:
import matplotlib.pyplot as plt

if len(matched) == 0:
    print("no matches to visualise yet. Check the CCOD file loaded and NB05 built the full spine.")
else:
    fig, ax = plt.subplots(2, 2, figsize=(13, 9))

    # A. funnel: from owner rows down to a confident match
    funnel = {
        "owner rows\nin CCOD": len(owners),
        "has a reg\nnumber": int(owners["company_reg_no"].notna().sum()),
        "matched to\nour dataset": len(matched),
    }
    ax[0, 0].bar(list(funnel.keys()), list(funnel.values()), color="#4477aa")
    ax[0, 0].set_title("From property owners to companies matched")
    for i, v in enumerate(funnel.values()):
        ax[0, 0].text(i, v, f"{v:,}", ha="center", va="bottom", fontsize=9)

    # B. matches by method (each maps to a confidence tier)
    mm = matched["method"].value_counts()
    ax[0, 1].barh(list(mm.index[::-1]), list(mm.values[::-1]), color="#228833")
    ax[0, 1].set_title("Matches by method (confidence tier)")
    for i, v in enumerate(mm.values[::-1]):
        ax[0, 1].text(v, i, f" {v:,}", va="center", fontsize=9)

    # C. top companies by number of properties owned
    top = con.execute("""
        SELECT c.company_name, count(*) AS n
        FROM signals s JOIN companies c ON c.company_number = s.company_number
        WHERE s.source='land_registry_ccod'
        GROUP BY c.company_name ORDER BY n DESC LIMIT 10
    """).df()
    if len(top):
        ax[1, 0].barh(top["company_name"][::-1], top["n"][::-1], color="#ccbb44")
        ax[1, 0].set_title("Top companies by number of properties")
        ax[1, 0].set_xlabel("properties owned")

    # D. matched properties by region
    reg = matched.assign(region=matched["region"].fillna("UNKNOWN")) \
                 .groupby("region").size().sort_values(ascending=False).head(10)
    if len(reg):
        ax[1, 1].barh(list(reg.index[::-1]), list(reg.values[::-1]), color="#ee6677")
        ax[1, 1].set_title("Matched properties by region (top 10)")
        ax[1, 1].tick_params(axis="y", labelsize=8)

    fig.suptitle("NB08 Land Registry: matching and the property signals produced", fontsize=13)
    fig.tight_layout()
    plt.show()

The funnel shows the rows narrowing from all property owners down to the ones we matched. Most matches come through the company number, which is why this source is more reliable than the name-only ones. The other charts show who owns the most and where the properties are.

This box saves and closes the database so the new signals are kept.

In [ ]:
con.close()
print("saved:", DB_PATH)

## Notes and what comes next
- CCOD covers England and Wales only. A company registered in Scotland or Northern Ireland only
  appears here if it owns property in England or Wales, so this signal under-reaches those firms.
- The registration number is the main join and is very reliable. Land Registry flags two known data
  quirks: some numbers have typing errors, and property bought before 1996 may have no number. Rows
  with no usable number fall back to the name ladder, and the `name_exact_unconfirmed` rows
  (confidence 0.6) are the ones to treat with care.
- A company can own many properties, so it can carry several `owns_property` rows. That is intended:
  the signals table is an event store. Group by `company_number` when you want one number per firm.
- Next signal source, same pattern (match, then write to `signals` with a confidence): **IPO trade
  marks** (holding a trade mark is a brand and growth signal). That source has no registration
  number and no API, so it is a name-and-postcode match like Contracts Finder.
- Naming note for the team: Sneha also uses notebook numbers in the 05-06 range and the news work
  uses 08-09 on another branch. Renumber if needed before merging to main.